In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

root = Path.cwd()
while not (root / "data" / "db").exists() and root != root.parent:
    root = root.parent
con = sqlite3.connect(root / "data" / "db" / "tennis.db")

by_year = pd.read_sql("SELECT * FROM serve_advantage_by_year", con)
pooled = pd.read_sql("SELECT * FROM serve_advantage_pooled", con)
print(by_year.shape, pooled.shape)
print(by_year["computed_on"].iloc[0])

In [ ]:
def plot_serve_by_year(events, filename="serve_advantage_by_year.png"):
    fig, ax = plt.subplots(figsize=(9, 5.4))
    n = len(events)
    for i, name in enumerate(events):
        d = by_year[by_year["name"] == name].sort_values("file_year")
        ax.errorbar(d["file_year"] + (i - (n - 1) / 2) * 0.15, d["serve_excess_pp"], yerr=d["half_width"],
                    marker="o", linestyle="none", capsize=3, label=name)

    ax.axhline(0, color="grey", linewidth=1)
    ax.set_xticks(range(2015, 2027))
    ax.set_ylabel("Service points won above expectation\n(percentage points)")
    ax.set_title("Serve advantage by tournament and year, 2015-2026")
    ax.legend(loc="lower left")
    fig.text(0.01, 0.012,
             "Data source: TennisMyLife — https://stats.tennismylife.org. Vertical bars: approximate 95% bootstrap\n"
             "interval over matches. Baseline: other events with the same surface and roof, same players, +/- 1 year.",
             fontsize=7, color="grey")
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    (root / "charts").mkdir(exist_ok=True)
    fig.savefig(root / "charts" / filename, dpi=200)
    plt.show()

plot_serve_by_year(["Madrid Masters", "Monte Carlo", "Miami Masters"])

In [ ]:
ranking = pd.read_sql("SELECT * FROM elo_ranking_snapshots", con)


def plot_ranking(as_of, surface="Overall", top=10, filename=None):
    d = ranking[(ranking["as_of"] == as_of) & (ranking["surface"] == surface)].sort_values("rank").head(top)
    d = d.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 5.4))
    ax.scatter(d["rating"], range(len(d)), zorder=3)
    for i_, (r_, m_) in enumerate(zip(d["rating"], d["matches_52w"])):
        ax.text(r_ + 6, i_, f"{int(r_)}  ({int(m_)} matches, last 52 weeks)", va="center", fontsize=8)
    ax.set_yticks(range(len(d)))
    ax.set_yticklabels(d["name"])
    ax.set_xlim(d["rating"].min() - 40, d["rating"].max() + 190)
    ax.grid(axis="x", alpha=0.3)
    label = "overall rating" if surface == "Overall" else f"{surface.lower()} rating"
    ax.set_title(f"Top {top} by Elo {label}, as of {as_of}")
    fig.text(0.01, 0.012,
             "Data source: TennisMyLife — https://stats.tennismylife.org. Elo rating calculated from ATP tour-level\n"
             "main-draw results since 2000: our own rating, not the official ATP ranking, and not a prediction.",
             fontsize=7, color="grey")
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    (root / "charts").mkdir(exist_ok=True)
    fig.savefig(root / "charts" / (filename or f"elo_{surface.lower()}_{as_of}.png"), dpi=200)
    plt.show()


plot_ranking("2024-12-22")

In [ ]:
p26 = pd.read_sql("SELECT * FROM elo_period_2026", con).sort_values("rank")


def plot_year_view(d, top=10, filename="elo_overall_2026.png"):
    as_of = d["as_of"].iloc[0]
    d = d.head(top).iloc[::-1].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(8.5, 5.6))
    for i_, row in d.iterrows():
        if pd.notna(row["elo_start"]):
            ax.plot([row["elo_start"], row["elo_end"]], [i_, i_], color="grey", linewidth=2, zorder=1)
            ax.scatter(row["elo_start"], i_, s=45, facecolor="white", edgecolor="grey", zorder=2)
            label = f"{int(row['elo_end'])}  ({int(row['change']):+d}, {int(row['matches_in_period'])} matches)"
        else:
            label = f"{int(row['elo_end'])}  (no rating on 1 Jan, {int(row['matches_in_period'])} matches)"
        ax.scatter(row["elo_end"], i_, s=55, zorder=3)
        ax.text(row["elo_end"] + 8, i_, label, va="center", fontsize=8)
    ax.set_yticks(range(len(d)))
    ax.set_yticklabels(d["name"])
    lows = d[["elo_start", "elo_end"]].min().min()
    ax.set_xlim(lows - 40, d["elo_end"].max() + 230)
    ax.grid(axis="x", alpha=0.3)
    ax.set_title(f"Elo overall rating, 1 Jan 2026 to {as_of}")
    fig.text(0.01, 0.012,
             "Data source: TennisMyLife — https://stats.tennismylife.org. Hollow dot: rating on 1 Jan 2026; filled dot: latest.\n"
             "Elo rating calculated from ATP tour-level main-draw results since 2000: our own rating, not the official ATP\n"
             "ranking, and not a prediction. Latest rating includes tournaments in progress.",
             fontsize=7, color="grey")
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    (root / "charts").mkdir(exist_ok=True)
    fig.savefig(root / "charts" / filename, dpi=200)
    plt.show()


plot_year_view(p26)

In [ ]:
def load(table):
    try:
        return pd.read_sql(f"SELECT * FROM {table}", con)
    except Exception:
        return None


tables = {t: load(t) for t in ["serve_advantage_by_year", "serve_advantage_pooled",
                               "elo_ranking_snapshots", "player_season_cards"]}
for t, df in tables.items():
    print(t, "->", "MISSING" if df is None else df.shape)

ranking = tables["elo_ranking_snapshots"]

In [ ]:
print(pd.read_sql("SELECT tourney_name, MIN(tourney_start) AS start, COUNT(*) AS matches FROM matches "
                  "WHERE event_no = '352s' AND file_year = 2026 GROUP BY tourney_name", con).to_string(index=False))

In [ ]:
tables["paris_characteristics_by_year"] = load("paris_characteristics_by_year")
tables["paris_vs_tour_pooled"] = load("paris_vs_tour_pooled")

PANELS = {
    "service_pts_won_pct": "Service points won (%)",
    "aces_pct": "Aces (% of service points)",
    "double_faults_pct": "Double faults (% of service points)",
    "first_serve_in_pct": "First serves in (%)",
    "service_games_held_pct": "Service games held (%)",
    "minutes_per_match_bo3": "Minutes per best-of-3 match",
}
LABELS = {**PANELS,
          "first_serve_pts_won_pct": "First-serve points won (%)",
          "second_serve_pts_won_pct": "Second-serve points won (%)",
          "matches_with_tiebreak_pct_bo3": "Best-of-3 matches with a tiebreak (%)"}
GROUP_STYLE = {"Paris Masters": {"color": "tab:red", "ls": "-"},
               "Rest of the tour": {"color": "grey", "ls": "-"},
               "Other indoor hard events": {"color": "tab:blue", "ls": "--"}}
CREDIT = "Data source: TennisMyLife — https://stats.tennismylife.org."


def plot_paris_vs_tour(metrics=None, filename="paris_vs_tour_by_year.png"):
    d = tables["paris_characteristics_by_year"]
    metrics = metrics or list(PANELS)
    ncols = 3
    nrows = -(-len(metrics) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 3.7 * nrows + 0.9), squeeze=False)
    for ax, m in zip(axes.ravel(), metrics):
        for g, st in GROUP_STYLE.items():
            s = d[(d["metric"] == m) & (d["group"] == g)].sort_values("file_year")
            if s.empty:
                continue
            if g == "Paris Masters":
                ax.errorbar(s["file_year"], s["value"],
                            yerr=[s["value"] - s["ci_low"], s["ci_high"] - s["value"]],
                            marker="o", capsize=2, color=st["color"], label=g)
            else:
                ax.plot(s["file_year"], s["value"], color=st["color"], linestyle=st["ls"], label=g)
                ax.fill_between(s["file_year"], s["ci_low"], s["ci_high"], color=st["color"], alpha=0.12)
        ax.set_title(LABELS[m], fontsize=10)
        ax.set_xticks(range(2015, int(d["file_year"].max()) + 1, 2))
        ax.grid(alpha=0.3)
    for ax in axes.ravel()[len(metrics):]:
        ax.axis("off")
    axes[0, 0].legend(fontsize=7, loc="best")
    fig.suptitle(f"Paris Masters compared with the rest of the tour, 2015-{int(d['file_year'].max())}", y=0.99)
    fig.text(0.01, 0.01,
             CREDIT + " Retired matches excluded. Bands and bars: approximate 95% bootstrap interval over matches. "
             "Rest of the tour = all other ATP tour-level events that year.\n"
             "Minutes per match: best-of-3 matches only. Service games held is derived from break points "
             "(rows with impossible counts excluded).",
             fontsize=7, color="grey")
    fig.tight_layout(rect=(0, 0.05, 1, 0.97))
    fig.savefig(root / "charts" / filename, dpi=200)
    plt.show()


def plot_paris_difference(filename="paris_vs_tour_difference.png"):
    d = tables["paris_vs_tour_pooled"]
    pct = d[d["unit"] == "percentage points"].sort_values("difference").reset_index(drop=True)
    mins = d[d["unit"] == "minutes"]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 5.4), gridspec_kw={"width_ratios": [3, 1]})
    a1.errorbar(pct["difference"], range(len(pct)),
                xerr=[pct["difference"] - pct["ci_low"], pct["ci_high"] - pct["difference"]], fmt="o", capsize=3)
    a1.axvline(0, color="grey", linewidth=1)
    a1.set_yticks(range(len(pct)))
    a1.set_yticklabels([LABELS[m] for m in pct["metric"]], fontsize=8)
    a1.set_xlabel("Paris minus the rest of the tour (percentage points)")
    a1.grid(axis="x", alpha=0.3)
    a2.errorbar(mins["difference"], [0],
                xerr=[mins["difference"] - mins["ci_low"], mins["ci_high"] - mins["difference"]], fmt="o", capsize=3)
    a2.axvline(0, color="grey", linewidth=1)
    a2.set_yticks([0])
    a2.set_yticklabels(["Minutes per\nbest-of-3 match"], fontsize=8)
    a2.set_xlabel("Paris minus the rest of the tour (minutes)")
    a2.grid(axis="x", alpha=0.3)
    yrs = d["years"].iloc[0]
    fig.suptitle(f"How Paris differs from the rest of the tour, completed editions {yrs}", y=0.99)
    fig.text(0.01, 0.01,
             CREDIT + " Bars: approximate 95% bootstrap interval over matches. Retired matches excluded. "
             "Rest of the tour = all other ATP tour-level events\nin the same years (all surfaces; Slams are best-of-5). "
             "Minutes and tiebreaks use best-of-3 matches only. Service games held excludes rows with impossible counts.",
             fontsize=7, color="grey")
    fig.tight_layout(rect=(0, 0.06, 1, 0.97))
    fig.savefig(root / "charts" / filename, dpi=200)
    plt.show()


plot_paris_vs_tour()
plot_paris_difference()

In [ ]:
tables["masters_service_games_held"] = load("masters_service_games_held")


def plot_masters_hold(filename="masters_service_games_held.png"):
    d = tables["masters_service_games_held"]
    ref = d[d["name"] == "All Masters 1000 (pooled)"].sort_values("file_year")
    names = sorted(n for n in d["name"].unique() if not n.startswith("All"))
    names = [n for n in names if n.startswith("Paris")] + [n for n in names if not n.startswith("Paris")]
    lo, hi = d["ci_low"].min() - 0.5, d["ci_high"].max() + 0.5

    fig, axes = plt.subplots(3, 3, figsize=(13, 10), sharey=True, sharex=True)
    for ax, nm in zip(axes.ravel(), names):
        s = d[d["name"] == nm].sort_values("file_year")
        ax.plot(ref["file_year"], ref["hold_pct"], color="grey", linestyle="--", linewidth=1, label="All Masters 1000")
        color = "tab:red" if nm.startswith("Paris") else "tab:blue"
        ax.errorbar(s["file_year"], s["hold_pct"], yerr=[s["hold_pct"] - s["ci_low"], s["ci_high"] - s["hold_pct"]],
                    marker="o", capsize=2, color=color)
        ax.set_title(nm, fontsize=10)
        ax.set_ylim(lo, hi)
        ax.set_xticks(range(2015, 2026, 2))
        ax.grid(alpha=0.3)
    axes[0, 0].legend(fontsize=7, loc="lower left")
    for ax in axes[:, 0]:
        ax.set_ylabel("Service games held (%)")
    fig.suptitle("Service games held at the Masters 1000 events, 2015-2025", y=0.995)
    fig.text(0.01, 0.01,
             "Data source: TennisMyLife — https://stats.tennismylife.org. Service games held derived from break points faced and saved; "
             "rows with impossible counts and retired matches excluded.\n"
             "Bars: approximate 95% bootstrap interval over matches. Dashed line: all nine Masters 1000 events pooled. "
             "Tournament-years with fewer than 20 matches are not shown (cancelled or short editions).",
             fontsize=7, color="grey")
    fig.tight_layout(rect=(0, 0.04, 1, 0.98))
    fig.savefig(root / "charts" / filename, dpi=200)
    plt.show()


plot_masters_hold()

In [ ]:
tables["paris_characteristics_by_year"] = load("paris_characteristics_by_year")
tables["paris_vs_tour_pooled"] = load("paris_vs_tour_pooled")


def plot_paris_hold(compare=False, show_average=True, filename="paris_service_games_held.png"):
    d = tables["paris_characteristics_by_year"]
    m = d[d["metric"] == "service_games_held_pct"]
    p = m[m["group"] == "Paris Masters"].sort_values("file_year")
    print(p[["file_year", "matches", "value"]].round(1).to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, 5.2))
    if compare:
        for g, (color, ls) in [("Rest of the tour", ("grey", "-")), ("Other indoor hard events", ("tab:blue", "--"))]:
            s = m[m["group"] == g].sort_values("file_year")
            ax.plot(s["file_year"], s["value"], color=color, linestyle=ls, linewidth=1.2, label=g)
    if show_average:
        avg = tables["paris_vs_tour_pooled"].query("metric == 'service_games_held_pct'")["paris"].iloc[0]
        ax.axhline(avg, color="tab:red", linestyle=":", linewidth=1, alpha=0.6,
                   label=f"Paris average 2015-2025 ({avg:.1f}%)")
    ax.plot(p["file_year"], p["value"], color="tab:red", linewidth=1.5, marker="o", markersize=10,
            markerfacecolor="tab:red", markeredgecolor="white", markeredgewidth=1.5, label="Paris Masters", zorder=3)
    for x, y in zip(p["file_year"], p["value"]):
        ax.annotate(f"{y:.1f}", (x, y), textcoords="offset points", xytext=(0, 11), ha="center",
                    fontsize=9, fontweight="bold")
    ax.set_xticks(range(2015, int(p["file_year"].max()) + 1))
    ax.set_ylabel("Service games held (%)")
    ax.set_title(f"Paris Masters: service games held by edition, 2015-{int(p['file_year'].max())}")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc="lower right")
    fig.text(0.01, 0.012,
             "Data source: TennisMyLife — https://stats.tennismylife.org. Service games held derived from break points faced and saved;\n"
             f"retired matches excluded. Each edition has {int(p['matches'].min())}-{int(p['matches'].max())} matches, "
             "so small year-to-year differences may be noise.",
             fontsize=7, color="grey")
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    fig.savefig(root / "charts" / filename, dpi=200)
    plt.show()


plot_paris_hold()

In [ ]:
import textwrap
import numpy as np
import matplotlib as mpl
from matplotlib.lines import Line2D

PAL = {"paris": "#E4002B", "tour": "#8D99AE", "indoor": "#0077B6", "accent": "#FF9F1C", "ink": "#1D3557"}
CRED = "Data source: TennisMyLife — https://stats.tennismylife.org."

mpl.rcParams.update({
    "font.size": 13, "axes.titlesize": 15, "axes.titleweight": "bold", "axes.labelsize": 13,
    "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.edgecolor": "#444444", "axes.labelcolor": PAL["ink"], "text.color": PAL["ink"],
})

PANELS = {
    "service_pts_won_pct": "Service points won (%)",
    "aces_pct": "Aces (% of service points)",
    "double_faults_pct": "Double faults (% of service points)",
    "first_serve_in_pct": "First serves in (%)",
    "service_games_held_pct": "Service games held (%)",
    "minutes_per_match_bo3": "Minutes per best-of-3 match",
}
LABELS = {**PANELS,
          "first_serve_pts_won_pct": "First-serve points won (%)",
          "second_serve_pts_won_pct": "Second-serve points won (%)",
          "matches_with_tiebreak_pct_bo3": "Best-of-3 matches with a tiebreak (%)"}
EXTRA_LABELS = {
    "break_points_converted_pct": "Break points converted (%)",
    "break_points_per_100_service_games": "Break points faced per 100 service games",
    "favourite_win_pct": "Favourite wins (%)",
    "avg_favourite_probability_pct": "Average favourite probability (%)",
    "favourite_wins_above_expected_pct": "Favourite wins minus expected (points)",
    "favourite_win_pct_50_60": "Favourite wins, 50-60% favourites (%)",
    "favourite_win_pct_60_75": "Favourite wins, 60-75% favourites (%)",
    "favourite_win_pct_75_plus": "Favourite wins, 75%+ favourites (%)",
    "matches_with_75_plus_favourite_pct": "Matches with a 75%+ favourite (%)",
}


def finish(fig, title, footer, filename, subtitle=None):
    w, h = fig.get_size_inches()
    wrapped = "\n".join(textwrap.fill(p, int(w * 12)) for p in footer.split("\n"))
    n = wrapped.count("\n") + 1
    fig.text(0.012, 0.15 / h, wrapped, fontsize=9.5, color="#444444", va="bottom", ha="left")
    fig.text(0.012, 1 - 0.1 / h, title, fontsize=21, fontweight="bold", color=PAL["ink"], va="top", ha="left")
    if subtitle:
        fig.text(0.012, 1 - 0.55 / h, subtitle, fontsize=13, color="#555555", va="top", ha="left")
    top = 1 - (1.05 if subtitle else 0.7) / h
    bottom = (0.25 + 0.16 * n) / h
    fig.tight_layout(rect=(0, bottom, 1, top))
    fig.savefig(root / "charts" / filename, dpi=200, facecolor="white")
    plt.show()


def dots(ax, x, y, lo, hi, color, interval=True, size=230):
    x, y, lo, hi = (np.asarray(v, dtype=float) for v in (x, y, lo, hi))
    if interval:
        ax.errorbar(x, y, xerr=[x - lo, hi - x], fmt="none", ecolor=color, elinewidth=2, capsize=4, alpha=0.7, zorder=2)
    clear = (lo > 0) | (hi < 0)
    ax.scatter(x[clear], y[clear], s=size, color=color, edgecolor="white", linewidth=1.5, zorder=3)
    ax.scatter(x[~clear], y[~clear], s=size, facecolor="white", edgecolor=color, linewidth=2.8, zorder=3)

In [ ]:
for t in ["paris_characteristics_by_year", "paris_vs_tour_pooled", "paris_extra_by_year", "paris_extra_pooled"]:
    tables[t] = load(t)


def plot_paris_hold(compare=False, show_average=True, filename="paris_service_games_held.png"):
    d = tables["paris_characteristics_by_year"]
    m = d[d["metric"] == "service_games_held_pct"]
    p = m[m["group"] == "Paris Masters"].sort_values("file_year")
    fig, ax = plt.subplots(figsize=(11, 6.6))
    if compare:
        for g, color, ls in [("Rest of the tour", PAL["tour"], "-"), ("Other indoor hard events", PAL["indoor"], "--")]:
            s = m[m["group"] == g].sort_values("file_year")
            ax.plot(s["file_year"], s["value"], color=color, linestyle=ls, linewidth=2.5, label=g)
    if show_average:
        avg = tables["paris_vs_tour_pooled"].query("metric == 'service_games_held_pct'")["paris"].iloc[0]
        ax.axhline(avg, color=PAL["accent"], linestyle="--", linewidth=2.5, label=f"Paris average 2015-2025: {avg:.1f}%")
    ax.plot(p["file_year"], p["value"], color=PAL["paris"], linewidth=3, marker="o", markersize=16,
            markeredgecolor="white", markeredgewidth=2, zorder=3, label="Paris Masters")
    for x, y in zip(p["file_year"], p["value"]):
        ax.annotate(f"{y:.1f}%", (x, y), textcoords="offset points", xytext=(0, 16), ha="center",
                    fontsize=14, fontweight="bold", color=PAL["paris"])
    ax.margins(y=0.18)
    ax.set_xticks(range(2015, int(p["file_year"].max()) + 1))
    ax.set_ylabel("Service games held (%)")
    ax.legend(loc="lower right", frameon=False)
    footer = (CRED + " Service games held is derived from break points faced and saved; retired matches excluded.\n"
              f"Each edition has {int(p['matches'].min())}-{int(p['matches'].max())} matches, so small year-to-year differences may be noise.")
    finish(fig, "Paris Masters: how often servers hold", footer, filename,
           subtitle=f"Share of service games held, by edition, 2015-{int(p['file_year'].max())}")


def plot_paris_vs_tour(metrics=None, filename="paris_vs_tour_by_year.png"):
    d = tables["paris_characteristics_by_year"]
    metrics = metrics or list(PANELS)
    nrows = -(-len(metrics) // 3)
    fig, axes = plt.subplots(nrows, 3, figsize=(16, 5.2 * nrows + 1.2), squeeze=False)
    for ax, m in zip(axes.ravel(), metrics):
        for g, color, ls in [("Rest of the tour", PAL["tour"], "-"), ("Other indoor hard events", PAL["indoor"], "--")]:
            s = d[(d["metric"] == m) & (d["group"] == g)].sort_values("file_year")
            ax.plot(s["file_year"], s["value"], color=color, linestyle=ls, linewidth=2.5, label=g)
        s = d[(d["metric"] == m) & (d["group"] == "Paris Masters")].sort_values("file_year")
        ax.plot(s["file_year"], s["value"], color=PAL["paris"], linewidth=3.5, marker="o", markersize=11,
                markeredgecolor="white", markeredgewidth=1.5, zorder=3, label="Paris Masters")
        ax.set_title(LABELS[m])
        ax.set_xticks(range(2015, 2026, 2))
    for ax in axes.ravel()[len(metrics):]:
        ax.axis("off")
    axes[0, 0].legend(loc="best", frameon=False)
    footer = (CRED + " Retired matches excluded. Rest of the tour = all other ATP tour-level events that year. "
              "Minutes per match: best-of-3 matches only. Service games held is derived from break points.\n"
              "Each Paris edition has 47-55 matches, so small year-to-year differences may be noise.")
    finish(fig, "Paris Masters vs the rest of the tour", footer, filename,
           subtitle="Evolution by edition, 2015-2025")


def plot_paris_difference(interval=True, filename="paris_vs_tour_difference.png"):
    d = tables["paris_vs_tour_pooled"]
    pct = d[d["unit"] == "percentage points"].sort_values("difference").reset_index(drop=True)
    mins = d[d["unit"] == "minutes"]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 7), gridspec_kw={"width_ratios": [3, 1]})
    dots(a1, pct["difference"], np.arange(len(pct)), pct["ci_low"], pct["ci_high"], PAL["paris"], interval)
    a1.axvline(0, color="black", linewidth=1.2)
    a1.set_yticks(range(len(pct)))
    a1.set_yticklabels([LABELS[m] for m in pct["metric"]], fontsize=13)
    a1.set_xlabel("Paris minus the rest of the tour (percentage points)")
    dots(a2, mins["difference"], np.array([0]), mins["ci_low"], mins["ci_high"], PAL["paris"], interval)
    a2.axvline(0, color="black", linewidth=1.2)
    a2.set_yticks([0])
    a2.set_yticklabels(["Minutes per\nbest-of-3 match"], fontsize=13)
    a2.set_xlabel("Paris minus the rest of the tour (minutes)")
    footer = (CRED + " Filled dot: the 95% interval excludes 0 (a clear difference). Hollow dot: it includes 0. "
              "Rest of the tour = all other ATP tour-level events in the same years (all surfaces; Slams are best-of-5).\n"
              "Retired matches excluded. Minutes use best-of-3 matches only. Service games held excludes rows with impossible counts.")
    finish(fig, "How Paris differs from the rest of the tour", footer, filename,
           subtitle=f"Completed editions {d['years'].iloc[0]}; each dot is Paris minus the average of the other events")


def plot_paris_extra_years(filename="paris_extra_by_year.png"):
    d = tables["paris_extra_by_year"]
    mets = ["break_points_converted_pct", "break_points_per_100_service_games",
            "favourite_win_pct", "favourite_wins_above_expected_pct"]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10.5))
    for ax, m in zip(axes.ravel(), mets):
        for g, color, ls in [("Rest of the tour", PAL["tour"], "-"), ("Other indoor hard events", PAL["indoor"], "--")]:
            s = d[(d["metric"] == m) & (d["group"] == g)].sort_values("file_year")
            ax.plot(s["file_year"], s["value"], color=color, linestyle=ls, linewidth=2.5, label=g)
        s = d[(d["metric"] == m) & (d["group"] == "Paris Masters")].sort_values("file_year")
        ax.plot(s["file_year"], s["value"], color=PAL["paris"], linewidth=3.5, marker="o", markersize=12,
                markeredgecolor="white", markeredgewidth=1.5, zorder=3, label="Paris Masters")
        for x, y in zip(s["file_year"], s["value"]):
            ax.annotate(f"{y:.1f}", (x, y), textcoords="offset points", xytext=(0, 10), ha="center",
                        fontsize=11, fontweight="bold", color=PAL["paris"])
        ax.set_title(EXTRA_LABELS[m])
        ax.set_xticks(range(2015, 2026, 2))
    axes[0, 0].legend(loc="best", frameon=False)
    footer = (CRED + " Break points: retired matches and rows with impossible counts excluded. "
              "Favourite = the player rated higher by our Elo v2 (not the ATP ranking); retirements count.\n"
              "Each Paris edition has 47-55 matches, so small year-to-year differences may be noise.")
    finish(fig, "Paris Masters: break points and favourites", footer, filename,
           subtitle="Evolution by edition, 2015-2025")


def plot_paris_extra_difference(interval=True, filename="paris_extra_difference.png"):
    d = tables["paris_extra_pooled"]
    pct = d[d["unit"] == "percentage points"]
    order = pct[pct["reference"] == "Rest of the tour"].sort_values("difference")["metric"].tolist()
    other = d[d["unit"] != "percentage points"]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 7.5), gridspec_kw={"width_ratios": [3, 1.1]})
    handles = []
    for k, (ref, color) in enumerate([("Rest of the tour", PAL["tour"]), ("Other indoor hard events", PAL["indoor"])]):
        s = pct[pct["reference"] == ref].set_index("metric").loc[order]
        pos = np.arange(len(order)) + (k - 0.5) * 0.3
        dots(a1, s["difference"], pos, s["ci_low"], s["ci_high"], color, interval, size=190)
        o = other[other["reference"] == ref]
        dots(a2, o["difference"], np.array([k]), o["ci_low"], o["ci_high"], color, interval, size=190)
        handles.append(Line2D([0], [0], marker="o", color="w", markerfacecolor=color, markersize=13,
                              label="vs " + ref.lower()))
    a1.axvline(0, color="black", linewidth=1.2)
    a1.set_yticks(range(len(order)))
    a1.set_yticklabels([EXTRA_LABELS[m] for m in order], fontsize=13)
    a1.set_xlabel("Paris minus the comparison group (percentage points)")
    a1.legend(handles=handles, loc="lower right", frameon=False)
    a2.axvline(0, color="black", linewidth=1.2)
    a2.set_ylim(-0.8, 1.8)
    a2.set_yticks([])
    a2.set_title("Break points faced\nper 100 service games", fontsize=13)
    a2.set_xlabel("Paris minus the comparison group")
    footer = (CRED + " Filled dot: the 95% interval excludes 0 (a clear difference). Hollow dot: it includes 0. "
              "Favourite = the player rated higher by our Elo v2 (not the ATP ranking).\n"
              "Rest of the tour = all other ATP tour-level events in the same years (all surfaces; Slams are best-of-5). "
              "Retired matches excluded from break points.")
    finish(fig, "How Paris differs: break points and favourites", footer, filename,
           subtitle=f"Completed editions {d['years'].iloc[0]}")


plot_paris_hold()
plot_paris_vs_tour()
plot_paris_difference()
plot_paris_extra_years()
plot_paris_extra_difference()

In [ ]:
tables["paris_characteristics_by_year"] = load("paris_characteristics_by_year")
tables["paris_vs_tour_pooled"] = load("paris_vs_tour_pooled")


def plot_paris_aces(compare=False, filename="paris_aces_per_service_point.png"):
    d = tables["paris_characteristics_by_year"]
    m = d[d["metric"] == "aces_pct"]
    p = m[m["group"] == "Paris Masters"].sort_values("file_year")
    pooled = tables["paris_vs_tour_pooled"].query("metric == 'aces_pct'").iloc[0]
    avg, rest = pooled["paris"], pooled["rest_of_tour"]

    fig, ax = plt.subplots(figsize=(11, 6.6))
    if compare:
        s = m[m["group"] == "Rest of the tour"].sort_values("file_year")
        ax.plot(s["file_year"], s["value"], color=PAL["tour"], linewidth=2.5, label="Rest of the tour")
    ax.axhline(avg, color=PAL["accent"], linestyle="--", linewidth=2.5, label=f"Paris average 2015-2025: {avg:.1f}%")
    ax.plot(p["file_year"], p["value"], color=PAL["paris"], linewidth=3, marker="o", markersize=16,
            markeredgecolor="white", markeredgewidth=2, zorder=3, label="Paris Masters")
    for x, y in zip(p["file_year"], p["value"]):
        ax.annotate(f"{y:.1f}%", (x, y), textcoords="offset points", xytext=(0, 16), ha="center",
                    fontsize=14, fontweight="bold", color=PAL["paris"])
    ax.margins(y=0.18)
    ax.set_xticks(range(2015, int(p["file_year"].max()) + 1))
    ax.set_ylabel("Aces (% of service points)")
    ax.legend(loc="lower right", frameon=False)
    footer = (CRED + " Retired matches excluded. Rest of the tour = all other ATP tour-level events in the same years "
              f"({rest:.1f}%, about 1 in {round(100 / rest)}).\n"
              f"Each edition has {int(p['matches'].min())}-{int(p['matches'].max())} matches, so small year-to-year differences may be noise.")
    finish(fig, f"Paris Masters, 2015-2025: about 1 in {round(100 / avg)} service points was an ace", footer, filename,
           subtitle=f"Share of service points that are aces, by edition, 2015-{int(p['file_year'].max())}")


plot_paris_aces()

In [ ]:
tables["aces_hard_events_pooled"] = load("aces_hard_events_pooled")
tables["aces_paris_vs_hard_by_year"] = load("aces_paris_vs_hard_by_year")


def plot_hard_aces_ranking(filename="paris_aces_vs_hard_events.png"):
    d = tables["aces_hard_events_pooled"]
    oth = d[d["name"].str.startswith("Other")].iloc[0]
    ev = d[~d["name"].str.startswith("Other")].sort_values("aces_pct").reset_index(drop=True)
    paris = ev[ev["name"] == "Paris Masters"].iloc[0]
    rank = int((ev["aces_pct"] > paris["aces_pct"]).sum()) + 1

    fig, ax = plt.subplots(figsize=(12, 7.4))
    ax.axvline(oth["aces_pct"], color=PAL["accent"], linestyle="--", linewidth=2.5, zorder=1)
    for i, r in ev.iterrows():
        is_paris = r["name"] == "Paris Masters"
        marker = "s" if r["indoor_outdoor"] == "I" else "o"
        clear = (r["ci_low"] > 0) or (r["ci_high"] < 0)
        color = PAL["paris"] if is_paris else PAL["tour"]
        filled = is_paris or clear
        ax.scatter(r["aces_pct"], i, s=520 if is_paris else 330, marker=marker,
                   facecolor=color if filled else "white", edgecolor=color if not is_paris else "white",
                   linewidth=3, zorder=3)
        ax.text(r["aces_pct"] + 0.18, i, f"{r['aces_pct']:.1f}%  (1 in {round(100 / r['aces_pct'])})",
                va="center", fontsize=15 if is_paris else 13, fontweight="bold" if is_paris else "normal",
                color=PAL["paris"] if is_paris else PAL["ink"])
    ax.set_yticks(range(len(ev)))
    ax.set_yticklabels(ev["name"], fontsize=14)
    for lab in ax.get_yticklabels():
        if lab.get_text() == "Paris Masters":
            lab.set_color(PAL["paris"])
            lab.set_fontweight("bold")
    ax.set_xlim(ev["aces_pct"].min() - 1, ev["aces_pct"].max() + 2.6)
    ax.set_xlabel("Aces (% of service points)")
    ax.grid(axis="y", alpha=0)
    handles = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=PAL["paris"], markersize=15, label="Paris Masters"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor=PAL["tour"], markersize=13,
               label="Clearly different from Paris"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="white", markeredgecolor=PAL["tour"],
               markeredgewidth=2.5, markersize=13, label="Not clearly different from Paris"),
        Line2D([0], [0], marker="s", color="w", markerfacecolor="#999999", markersize=13, label="Indoor event"),
        Line2D([0], [0], color=PAL["accent"], linestyle="--", linewidth=2.5,
               label=f"Other hard-court events: {oth['aces_pct']:.1f}%"),
    ]
    ax.legend(handles=handles, loc="lower right", frameon=False, fontsize=11)
    footer = (CRED + " Hard-court events among the 14 core ATP events; completed editions 2015-2025; retired matches excluded. "
              "Filled dot: the 95% interval of the difference from Paris excludes 0 (bootstrap over matches).\n"
              "Raw ace rates mix the court with who plays and the format (Slams are best-of-5). "
              "Only the ATP Finals is another indoor hard event in this group (163 matches in total).")
    finish(fig, f"Paris ranks #{rank} of {len(ev)} hard-court events for aces", footer, filename,
           subtitle="Share of service points that are aces, hard-court events only, 2015-2025")


def plot_paris_vs_hard_years(filename="paris_aces_vs_hard_by_year.png"):
    d = tables["aces_paris_vs_hard_by_year"]
    p = d[d["group"] == "Paris Masters"].sort_values("file_year")
    o = d[d["group"] == "Other hard-court events"].sort_values("file_year")
    fig, ax = plt.subplots(figsize=(11, 6.6))
    ax.plot(o["file_year"], o["aces_pct"], color=PAL["tour"], linewidth=3, label="Other hard-court events")
    ax.plot(p["file_year"], p["aces_pct"], color=PAL["paris"], linewidth=3, marker="o", markersize=16,
            markeredgecolor="white", markeredgewidth=2, zorder=3, label="Paris Masters")
    for x, y in zip(p["file_year"], p["aces_pct"]):
        ax.annotate(f"{y:.1f}%", (x, y), textcoords="offset points", xytext=(0, 16), ha="center",
                    fontsize=14, fontweight="bold", color=PAL["paris"])
    ax.margins(y=0.18)
    ax.set_xticks(range(2015, int(p["file_year"].max()) + 1))
    ax.set_ylabel("Aces (% of service points)")
    ax.legend(loc="lower right", frameon=False)
    footer = (CRED + " Hard-court events among the 14 core ATP events; retired matches excluded. "
              f"Each Paris edition has {int(p['matches'].min())}-{int(p['matches'].max())} matches, so small year-to-year differences may be noise.")
    finish(fig, "Paris Masters vs the other hard-court events: aces", footer, filename,
           subtitle="Share of service points that are aces, by edition, 2015-2025")


plot_hard_aces_ranking()
plot_paris_vs_hard_years()